In [ ]:
import os
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# figure 2

# setup paths and load data
drive_data_dir = "/content/drive/MyDrive/data"
parquet_path = os.path.join(drive_data_dir, "cleaned_patent_panel.parquet")

con = duckdb.connect()
df_plot = con.execute(f"""
    SELECT
        vc_backed,
        npl_ratio,
        LN(total_claims + 1) AS log_claims
    FROM '{parquet_path}'
    WHERE grant_year BETWEEN 2000 AND 2020;
""").df()

df_plot['vc_status'] = df_plot['vc_backed'].map({1: 'VC-Backed', 0: 'Non-VC'})

# styling configuration
plt.rcParams.update({
    'font.size': 10,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#000000',
    'grid.color': '#000000',
    'grid.alpha': 0.25
})

fig, ax = plt.subplots(figsize=(8.5, 5), dpi=300)

sns.kdeplot(
    data=df_plot,
    x='log_claims',
    y='npl_ratio',
    hue='vc_status',
    palette={'VC-Backed': '#4da6ff', 'Non-VC': '#005a32'},
    fill=True,
    alpha=0.4,
    levels=5,
    ax=ax
)

ax.set_title('Figure 2: Joint Distribution of Patent Scope and Science Linkage by Funding Status', fontsize=11, fontweight='bold', pad=14)
ax.set_xlabel('Log Total Claims (Scope)', fontsize=10, labelpad=8)
ax.set_ylabel('NPL Ratio (Science Linkage)', fontsize=10, labelpad=8)
ax.grid(True, linestyle=':')

plt.tight_layout()

fig2_path = os.path.join(drive_data_dir, "figure2_claims_npl.png")
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.show()